# Test 7: Imbalanced Class Evaluation (Real-World Deployment Simulation)

**Goal**: Simulate a realistic deployment scenario where the class ratio is **90% safe / 10% malicious** instead of the 50/50 training balance.

**Why this matters**: A model trained on balanced data can have inflated accuracy when evaluated on balanced data. In real APK scanning the vast majority of apps are safe. Reviewers will ask *"does this hold up in practice?"*.

**Method**: No retraining needed. We resample the existing validation set to approximate a 90/10 ratio by keeping all malicious samples and downsampling safe ones. We then report Precision / Recall / F1 / FPR at the optimal threshold.

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin`
- `/kaggle/input/model-codebert/model_codebert.bin`

**Outputs**: `test7_imbalanced_results.txt`, `test7_precision_recall_bar.png`

**Runtime**: ~15–20 min on GPU (inference only)

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 11.7 MB/s eta 0:00:0000:01


In [2]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import RobertaConfig, RobertaModel, AutoTokenizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

Imports OK
CUDA: True


In [3]:
class Args:
    train_file       = '/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl'
    gcb_weights      = '/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin'
    codebert_weights = '/kaggle/input/datasets/hasanmahmudabdullah/codebert/model_codebert.bin'
    code_length      = 384
    data_flow_length = 128
    eval_batch_size  = 32
    seed             = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_gpu  = torch.cuda.device_count()
args = Args()

OPT_THRESHOLD = 0.45  # best F1 threshold from Test 2
SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if args.n_gpu > 0: torch.cuda.manual_seed_all(s)
set_seed(SEED)
print(f'Device: {args.device}  |  Threshold: {OPT_THRESHOLD}')

Device: cuda  |  Threshold: 0.45


In [4]:
class ModelA(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder = encoder; self.config = config
        self.tokenizer = tokenizer; self.args = args
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)
    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext_mask = (1.0 - attn_mask) * -10000.0
        ext_mask = ext_mask.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext_mask,
            head_mask=[None]*self.config.num_hidden_layers)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

class ModelB(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder = encoder; self.config = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

In [5]:
def load_valid_lines(file_path):
    """Load and pre-filter JSONL: skip blank, non-UTF-8, and non-dict lines.
    This prevents JSONDecodeError and AttributeError in DataLoader workers."""
    valid = []
    skipped = 0
    with open(file_path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                skipped += 1
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and 'code' in obj:
                    valid.append(line)
                else:
                    skipped += 1
            except Exception:
                skipped += 1
    print(f'  Loaded {len(valid):,} valid lines  ({skipped:,} skipped)')
    return valid


class TextDataset(Dataset):
    """GraphCodeBERT dataset — DFG-aware."""
    def __init__(self, tokenizer, args, file_path):
        self.args = args; self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        self.lines = load_valid_lines(file_path)  # pre-filtered

    def __len__(self): return len(self.lines)

    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    def __getitem__(self, idx):
        entry      = json.loads(self.lines[idx])  # always safe — pre-filtered
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)
        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True)
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        n2t, p2n = {}, {}
        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk = (sp[0], sp[1], ep[0], ep[1]); p2n[pk] = ni
            try:
                cs = self._char_idx(code_lines, sp)
                ce = self._char_idx(code_lines, ep)
                n2t[ni] = [i for i, (ts, te) in enumerate(offsets)
                           if ts != te and ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
            except IndexError:
                n2t[ni] = []
        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len, :c_len] = True
        for ni, node in enumerate(dfg):
            ani = c_len + ni
            for ti in n2t.get(ni, []): mask[ani, ti] = mask[ti, ani] = True
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n: ap = c_len + p2n[pk]; mask[ani, ap] = mask[ap, ani] = True
            mask[ani, ani] = True
        ids   = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad   = self.total_len - len(ids)
        if pad > 0: ids += [self.tokenizer.pad_token_id] * pad; p_ids += [1] * pad
        return {
            'input_ids': torch.tensor(ids,   dtype=torch.long),
            'p_ids':     torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(mask,  dtype=torch.float),
            'label':     torch.tensor(label, dtype=torch.long)
        }


class SimpleCodeDataset(Dataset):
    """CodeBERT dataset — text only."""
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer = tokenizer; self.args = args
        self.lines = load_valid_lines(file_path)  # pre-filtered

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        entry = json.loads(self.lines[idx])  # always safe — pre-filtered
        code  = entry.get('code', '')
        label = int(entry.get('label', 0))
        tok   = self.tokenizer(code, max_length=self.args.code_length,
                               truncation=True, padding='max_length')
        return {
            'input_ids':      torch.tensor(tok['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label':          torch.tensor(label,                 dtype=torch.long)
        }

In [6]:
print('Loading Model A (GraphCodeBERT)...')
cfg_a = RobertaConfig.from_pretrained('microsoft/graphcodebert-base'); cfg_a.num_labels = 2
tok_a = AutoTokenizer.from_pretrained('microsoft/graphcodebert-base')
enc_a = RobertaModel.from_pretrained('microsoft/graphcodebert-base', config=cfg_a)
model_a = ModelA(enc_a, cfg_a, tok_a, args).to(args.device)
model_a.load_state_dict(torch.load(args.gcb_weights, map_location=args.device))
model_a.eval(); print('  ✓ Model A loaded')

has_b = os.path.exists(args.codebert_weights)
if has_b:
    print('Loading Model B (CodeBERT)...')
    cfg_b = RobertaConfig.from_pretrained('microsoft/codebert-base'); cfg_b.num_labels = 2
    tok_b = AutoTokenizer.from_pretrained('microsoft/codebert-base')
    enc_b = RobertaModel.from_pretrained('microsoft/codebert-base', config=cfg_b)
    model_b = ModelB(enc_b, cfg_b, tok_b, args).to(args.device)
    model_b.load_state_dict(torch.load(args.codebert_weights, map_location=args.device))
    model_b.eval(); print('  ✓ Model B loaded')
else:
    print(f'  ⚠ CodeBERT not found at {args.codebert_weights} — ensemble skipped')
    model_b = None

Loading Model A (GraphCodeBERT)...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  ✓ Model B loaded


In [7]:
# Build datasets — uses same 90/10 val split as main training notebook
print('Building TextDataset A...')
ds_a    = TextDataset(tok_a, args, args.train_file)
n       = len(ds_a)
train_n = int(0.9 * n); val_n = n - train_n
gen     = torch.Generator().manual_seed(SEED)
_, val_ds_a = random_split(ds_a, [train_n, val_n], generator=gen)
print(f'Val samples: {val_n:,}')

if model_b:
    print('Building SimpleCodeDataset B...')
    ds_b = SimpleCodeDataset(tok_b, args, args.train_file)
    gen  = torch.Generator().manual_seed(SEED)
    _, val_ds_b = random_split(ds_b, [train_n, val_n], generator=gen)

Building TextDataset A...
  Loaded 199,960 valid lines  (0 skipped)
Val samples: 19,996
Building SimpleCodeDataset B...
  Loaded 199,960 valid lines  (0 skipped)


In [8]:
@torch.no_grad()
def get_probs_a(model, dataset):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    all_p, all_l = [], []
    for batch in tqdm(loader, desc='Inference A'):
        inp = {k: batch[k].to(args.device) for k in ('input_ids', 'p_ids', 'attn_mask')}
        pr  = model(**inp)[:, 1].cpu().numpy()
        all_p.extend(pr); all_l.extend(batch['label'].numpy())
    return np.array(all_p), np.array(all_l)

@torch.no_grad()
def get_probs_b(model, dataset):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    all_p, all_l = [], []
    for batch in tqdm(loader, desc='Inference B'):
        inp = {k: batch[k].to(args.device) for k in ('input_ids', 'attention_mask')}
        pr  = model(**inp)[:, 1].cpu().numpy()
        all_p.extend(pr); all_l.extend(batch['label'].numpy())
    return np.array(all_p), np.array(all_l)

probs_a, labels_bal = get_probs_a(model_a, val_ds_a)
if model_b:
    probs_b, _ = get_probs_b(model_b, val_ds_b)
    probs_ens  = (probs_a + probs_b) / 2.0
print('Inference complete')
print(f'Balanced set: {len(labels_bal):,} samples  |  malicious ratio: {labels_bal.mean():.4f}')

Inference B: 100%|██████████| 625/625 [03:47<00:00,  2.75it/s]

Inference complete
Balanced set: 19,996 samples  |  malicious ratio: 0.4951


In [9]:
# Resample to 90% safe / 10% malicious
# Keep ALL malicious; downsample safe to 9x that count
rng      = np.random.default_rng(SEED)
mal_idx  = np.where(labels_bal == 1)[0]
safe_idx = np.where(labels_bal == 0)[0]

n_mal_target = len(safe_idx) // 9          # keep all safe, take 1/9th of malicious
mal_sample   = rng.choice(mal_idx, size=n_mal_target, replace=False)
imb_idx      = np.concatenate([safe_idx, mal_sample])


labels_imb  = labels_bal[imb_idx]
probs_a_imb = probs_a[imb_idx]
if model_b:
    probs_ens_imb = probs_ens[imb_idx]

print(f'Imbalanced set: {len(labels_imb):,} samples')
print(f'  Malicious: {labels_imb.sum():,}  ({labels_imb.mean()*100:.1f}%)')
print(f'  Safe:      {(labels_imb==0).sum():,}  ({(1-labels_imb.mean())*100:.1f}%)')

Imbalanced set: 11,216 samples
  Malicious: 1,121  (10.0%)
  Safe:      10,095  (90.0%)


In [10]:
print('hello')

hello


In [11]:
def evaluate(probs, labels, name, threshold=OPT_THRESHOLD):
    preds = (probs >= threshold).astype(int)
    acc   = accuracy_score(labels, preds)
    prec  = precision_score(labels, preds, zero_division=0)
    rec   = recall_score(labels, preds, zero_division=0)
    f1    = f1_score(labels, preds, zero_division=0)
    auc_  = roc_auc_score(labels, probs)
    pr_a  = average_precision_score(labels, probs)
    cm    = confusion_matrix(labels, preds)
    fn    = cm[1, 0] if cm.shape == (2, 2) else 0
    fp    = cm[0, 1] if cm.shape == (2, 2) else 0
    fpr   = fp / max(cm[0, 0] + cm[0, 1], 1)
    print(f'\n=== {name} (threshold={threshold}) ===')
    print(f'  Accuracy  : {acc:.4%}')
    print(f'  Precision : {prec:.4f}  (of flagged apps, how many are actually malicious)')
    print(f'  Recall    : {rec:.4f}  (of all malicious, how many we caught)')
    print(f'  F1        : {f1:.4f}')
    print(f'  ROC-AUC   : {auc_:.4f}')
    print(f'  PR-AUC    : {pr_a:.4f}')
    print(f'  FN        : {fn:,}  (missed malware)')
    print(f'  FP        : {fp:,}  (false alarms on safe apps)')
    print(f'  FPR       : {fpr:.4f}  (false alarm rate)')
    return dict(name=name, acc=acc, prec=prec, rec=rec, f1=f1,
                auc=auc_, pr_auc=pr_a, fn=fn, fp=fp, fpr=fpr)

In [12]:
print('\n=== BALANCED (50/50) ===')
res_bal_a = evaluate(probs_a, labels_bal, 'GraphCodeBERT+DFG [balanced 50/50]')
if model_b:
    res_bal_ens = evaluate(probs_ens, labels_bal, 'Ensemble [balanced 50/50]')


=== BALANCED (50/50) ===

=== GraphCodeBERT+DFG [balanced 50/50] (threshold=0.45) ===
  Accuracy  : 91.7483%
  Precision : 0.9020  (of flagged apps, how many are actually malicious)
  Recall    : 0.9350  (of all malicious, how many we caught)
  F1        : 0.9182
  ROC-AUC   : 0.9798
  PR-AUC    : 0.9797
  FN        : 644  (missed malware)
  FP        : 1,006  (false alarms on safe apps)
  FPR       : 0.0997  (false alarm rate)

=== Ensemble [balanced 50/50] (threshold=0.45) ===
  Accuracy  : 91.5333%
  Precision : 0.8859  (of flagged apps, how many are actually malicious)
  Recall    : 0.9516  (of all malicious, how many we caught)
  F1        : 0.9176
  ROC-AUC   : 0.9804
  PR-AUC    : 0.9803
  FN        : 479  (missed malware)
  FP        : 1,214  (false alarms on safe apps)
  FPR       : 0.1203  (false alarm rate)


In [13]:
print('\n=== IMBALANCED (90% safe / 10% malicious) ===')
res_imb_a = evaluate(probs_a_imb, labels_imb, 'GraphCodeBERT+DFG [imbalanced 90/10]')
if model_b:
    res_imb_ens = evaluate(probs_ens_imb, labels_imb, 'Ensemble [imbalanced 90/10]')


=== IMBALANCED (90% safe / 10% malicious) ===

=== GraphCodeBERT+DFG [imbalanced 90/10] (threshold=0.45) ===
  Accuracy  : 90.3442%
  Precision : 0.5093  (of flagged apps, how many are actually malicious)
  Recall    : 0.9313  (of all malicious, how many we caught)
  F1        : 0.6585
  ROC-AUC   : 0.9782
  PR-AUC    : 0.8851
  FN        : 77  (missed malware)
  FP        : 1,006  (false alarms on safe apps)
  FPR       : 0.0997  (false alarm rate)

=== Ensemble [imbalanced 90/10] (threshold=0.45) ===
  Accuracy  : 88.6145%
  Precision : 0.4657  (of flagged apps, how many are actually malicious)
  Recall    : 0.9438  (of all malicious, how many we caught)
  F1        : 0.6236
  ROC-AUC   : 0.9783
  PR-AUC    : 0.8861
  FN        : 63  (missed malware)
  FP        : 1,214  (false alarms on safe apps)
  FPR       : 0.1203  (false alarm rate)


In [14]:
print('\nThreshold sensitivity — GraphCodeBERT+DFG on imbalanced 90/10 set:')
print(f'{"Thresh":>7} {"Prec":>7} {"Rec":>7} {"F1":>7} {"FPR":>7} {"FN":>7}')
print('-' * 50)
for th in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    p    = (probs_a_imb >= th).astype(int)
    prec = precision_score(labels_imb, p, zero_division=0)
    rec  = recall_score(labels_imb,    p, zero_division=0)
    f1   = f1_score(labels_imb,        p, zero_division=0)
    cm   = confusion_matrix(labels_imb, p)
    fn   = cm[1, 0] if cm.shape == (2, 2) else 0
    fp   = cm[0, 1] if cm.shape == (2, 2) else 0
    fpr  = fp / max(cm[0, 0] + cm[0, 1], 1)
    marker = ' <-- optimal' if th == OPT_THRESHOLD else ''
    print(f'{th:>7.2f} {prec:>7.4f} {rec:>7.4f} {f1:>7.4f} {fpr:>7.4f} {fn:>7,}{marker}')


Threshold sensitivity — GraphCodeBERT+DFG on imbalanced 90/10 set:
 Thresh    Prec     Rec      F1     FPR      FN
--------------------------------------------------
   0.30  0.4142  0.9688  0.5803  0.1522      35
   0.35  0.4374  0.9599  0.6009  0.1371      45
   0.40  0.4671  0.9438  0.6249  0.1196      63
   0.45  0.5093  0.9313  0.6585  0.0997      77 <-- optimal
   0.50  0.5573  0.9063  0.6902  0.0799     105
   0.55  0.6151  0.8840  0.7255  0.0614     130
   0.60  0.6810  0.8626  0.7611  0.0449     154
   0.65  0.7284  0.8421  0.7811  0.0349     177


In [15]:
has_ens = model_b is not None
if has_ens:
    prec_v  = [res_bal_a['prec'], res_imb_a['prec'], res_bal_ens['prec'], res_imb_ens['prec']]
    rec_v   = [res_bal_a['rec'],  res_imb_a['rec'],  res_bal_ens['rec'],  res_imb_ens['rec']]
    f1_v    = [res_bal_a['f1'],   res_imb_a['f1'],   res_bal_ens['f1'],   res_imb_ens['f1']]
    xlbls   = ['GCB+DFG\nBalanced', 'GCB+DFG\nImbalanced', 'Ensemble\nBalanced', 'Ensemble\nImbalanced']
else:
    prec_v  = [res_bal_a['prec'], res_imb_a['prec']]
    rec_v   = [res_bal_a['rec'],  res_imb_a['rec']]
    f1_v    = [res_bal_a['f1'],   res_imb_a['f1']]
    xlbls   = ['GCB+DFG\nBalanced', 'GCB+DFG\nImbalanced']

x = np.arange(len(xlbls)); w = 0.26
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w, prec_v, w, label='Precision', color='#4C72B0', alpha=0.88)
b2 = ax.bar(x,     rec_v,  w, label='Recall',    color='#55A868', alpha=0.88)
b3 = ax.bar(x + w, f1_v,   w, label='F1',        color='#DD8452', alpha=0.88)
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)
ax.set_ylim(0.0, 1.12); ax.set_xticks(x); ax.set_xticklabels(xlbls, fontsize=10)
ax.set_ylabel('Score', fontsize=13)
ax.set_title(f'Precision / Recall / F1: Balanced vs Imbalanced (threshold={OPT_THRESHOLD})', fontsize=13)
ax.legend(fontsize=11); ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
out_img = '/kaggle/working/test7_precision_recall_bar.png'
fig.savefig(out_img, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test7_precision_recall_bar.png


In [16]:
out_txt = '/kaggle/working/test7_imbalanced_results.txt'
with open(out_txt, 'w') as fh:
    fh.write('Test 7: Imbalanced Class Evaluation\n')
    fh.write('=' * 60 + '\n')
    fh.write(f'Threshold: {OPT_THRESHOLD}  |  Ratio: 90% safe / 10% malicious\n\n')
    rows = [res_bal_a, res_imb_a] + ([res_bal_ens, res_imb_ens] if has_ens else [])
    for res in rows:
        fh.write(f"\n{res['name']}\n")
        for k in ['acc', 'prec', 'rec', 'f1', 'auc', 'pr_auc', 'fn', 'fp', 'fpr']:
            v = res[k]
            fh.write(f'  {k:<8}: {v:.4f}\n' if isinstance(v, float) else f'  {k:<8}: {v:,}\n')
print(f'Saved -> {out_txt}')
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test7_imbalanced_results.txt
Saved -> /kaggle/working/test7_precision_recall_bar.png
